# Crawling Data Detik.com

Pada tahap ini dilakukan proses pengumpulan data berita secara otomatis (*web crawling*) dari portal **Detik.com**. Data yang diambil berupa judul dan isi berita dari dua kategori, yaitu **SPORT** dan **FINANCE**.

Beberapa *library* Python digunakan untuk mendukung proses pengambilan dan pengolahan data, yaitu:

- **Requests** digunakan untuk mengakses halaman web dan mengambil kode HTML.
- **BeautifulSoup** digunakan untuk membaca struktur HTML serta mengambil elemen yang diperlukan, seperti judul dan URL berita.
- **Pandas** digunakan untuk mengolah hasil *crawling* dan menyusunnya ke dalam bentuk tabel (*DataFrame*).
- **Trafilatura** digunakan untuk mengambil teks utama dari halaman berita secara otomatis agar isi yang diperoleh lebih bersih dan relevan.



In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import trafilatura
import random
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer

# Jika belum ada, install dulu di terminal:
# pip install Sastrawi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# =========================================================
# 1. CRAWLING (bagian ini tidak diubah, hanya dirapikan)
# =========================================================

kategori_list = ['sport', 'finance']
target_per_kategori = 100
data_berita = []

headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/131.0.0.0 Safari/537.36'
    )
}

halaman_awal = {
    'sport': 6,
    'finance': 9
}

for kategori in kategori_list:

    print(f"\n=== Crawling {kategori.upper()} ===")

    halaman = halaman_awal[kategori]
    terkumpul = 0
    link_tersimpan = set()

    while terkumpul < target_per_kategori:

        url = f'https://{kategori}.detik.com/indeks?page={halaman}'

        try:
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code != 200:
                print(f"Halaman {halaman} gagal diakses ({response.status_code})")
                halaman += 1
                continue

            soup = BeautifulSoup(response.text, 'html.parser')
            articles = soup.find_all('article')

            if not articles:
                articles = soup.select('.list-content article, .media__text')

            if not articles:
                print(f"Tidak ada artikel pada halaman {halaman}.")
                halaman += 1
                continue

            random.shuffle(articles)

            for article in articles:

                if terkumpul >= target_per_kategori:
                    break

                title_tag = article.find(['h2', 'h3'])
                link_tag = article.find('a')

                if not title_tag or not link_tag:
                    continue

                tautan = link_tag.get('href')

                if not tautan or tautan in link_tersimpan:
                    continue

                judul = title_tag.get_text(strip=True)

                downloaded = trafilatura.fetch_url(tautan)
                if not downloaded:
                    continue

                isi_berita = trafilatura.extract(
                    downloaded,
                    include_comments=False,
                    include_tables=False
                )

                if not isi_berita:
                    continue

                link_tersimpan.add(tautan)

                data_berita.append({
                    'Kategori': kategori.upper(),
                    'Judul Berita': judul,
                    'Isi Berita': isi_berita
                })

                terkumpul += 1
                print(f"[{kategori.upper()}] {terkumpul}/{target_per_kategori} - {judul[:60]}")

                time.sleep(0.5)

            halaman += 1

        except Exception as e:
            print(f"Terjadi kesalahan pada halaman {halaman}: {e}")
            halaman += 1

        time.sleep(1.5)


df_berita = pd.DataFrame(data_berita)
df_berita.drop_duplicates(subset=['Judul Berita'], inplace=True)
df_berita = df_berita.sample(frac=1, random_state=190).reset_index(drop=True)

# Simpan dulu data mentahnya (opsional, buat cadangan/cek manual)
df_berita.to_csv('data_berita_sport_finance_saya.csv', index=False)

print("\n=== CRAWLING SELESAI ===")
print(f"Total data: {len(df_berita)} baris")
print(df_berita['Kategori'].value_counts())


# =========================================================
# 2. PREPROCESSING TEKS
# =========================================================

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def preprocessing_teks(teks):
    # Case folding
    teks = teks.lower()
    # Hapus angka
    teks = re.sub(r'\d+', '', teks)
    # Hapus tanda baca
    teks = teks.translate(str.maketrans('', '', string.punctuation))
    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    # Hapus stopword
    teks = stopword_remover.remove(teks)
    # Stemming (mengubah kata ke bentuk dasar)
    teks = stemmer.stem(teks)
    return teks

print("\n=== PREPROCESSING TEKS ===")
df_berita['Isi Bersih'] = df_berita['Isi Berita'].apply(preprocessing_teks)


# =========================================================
# 3. TF-IDF VECTORIZATION
# =========================================================

print("\n=== MEMBUAT TF-IDF ===")

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_berita['Isi Bersih'])

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# Tambahkan kolom label kategori di paling depan
df_tfidf.insert(0, 'Label Kategori', df_berita['Kategori'].values)

# Simpan hasil akhir dalam format matriks TF-IDF
nama_file_tfidf = 'tfidf_berita_saya.csv'
df_tfidf.to_csv(nama_file_tfidf, index=False)

print("\n=== PROSES SELESAI ===")
print(f"File TF-IDF tersimpan sebagai: {nama_file_tfidf}")
print(f"Ukuran matriks: {df_tfidf.shape[0]} baris x {df_tfidf.shape[1]} kolom")


=== Crawling SPORT ===
[SPORT] 1/100 - Tekad Alwi Farhan Lampaui Batas di Asian Games 2026
[SPORT] 2/100 - Nova Widianto Mulai Tangani Ganda Campuran Pelatnas PBSI
[SPORT] 3/100 - MotoGP San Marino 2026: Jaga Puncak Klasemen Bukan Prioritas
[SPORT] 4/100 - HUT TNI Akan Dimeriahkan Kejuaraan Tingkat Nasional di 8 Cab
[SPORT] 5/100 - Menuju Asian Games 2026, Ketum KOI Tekankan Kolaborasi
[SPORT] 6/100 - Setelah 9 Tahun Gelar Basket, Kini LJK 2026 Rambah Padel
[SPORT] 7/100 - IHR 2026 Perluas Olahraga Pacuan Kuda Indonesia
[SPORT] 8/100 - Timnas Basket Putra di Asian Games 2026, Menangi Laga Pertam
[SPORT] 9/100 - Ambisi Morgan Holindo Jadi Juara Nasional Eshark Rok Cup 202
[SPORT] 10/100 - WRT 32 di Lone Star Le Mans: Startnya Sudah Bagus, tapi...
[SPORT] 11/100 - Ganda Campuran Indonesia Kembali Rombak Pemain
[SPORT] 12/100 - Ketum KOI dan Menpora Mengukuhkan Tim Indonesia untuk Asian 
[SPORT] 13/100 - Asian Games 2026: Banjir Landa Nagoya, Menpora Yakin Jepang 
[SPORT] 14/100 - CdM To

Hasil pada tabel di atas menunjukkan data berita yang telah berhasil dikumpulkan dari halaman indeks. Data tersebut masih berupa data awal sehingga perlu melalui proses pembersihan dan pengolahan teks sebelum digunakan pada tahap berikutnya, yaitu *text preprocessing* dalam proses Web Mining.